# Evaluate -- phase 5

Phase 4 trained one model and eyeballed the score. This notebook makes
the evaluation defensible:

1. reload `featured/` rather than re-deriving the frame
2. score a baseline, so rmse has something to beat
3. train two configurations and compare them on the same holdout
4. confirm both are reproducible across a re-run

The two runs here are what phase 6 replays into MLflow.

## 1. Setup

Same bucket as phase 4. `featured/hour.parquet` already exists, so the
raw CSV is not touched.

In [1]:
import io

import boto3
import numpy as np
import pandas as pd

REGION = "ca-central-1"

# From `terraform -chdir=infra/domain output -raw data_bucket`.
BUCKET = "sagemaker-domain-dev-data-pqkx2l"

FEATURED_KEY = "featured/hour.parquet"

s3 = boto3.client("s3", region_name=REGION)

obj = s3.get_object(Bucket=BUCKET, Key=FEATURED_KEY)
df = pd.read_parquet(io.BytesIO(obj["Body"].read()))

print(f"{len(df)} rows x {df.shape[1]} cols from s3://{BUCKET}/{FEATURED_KEY}")
df.head()

17379 rows x 13 cols from s3://sagemaker-domain-dev-data-pqkx2l/featured/hour.parquet


,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,cnt
0,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,16
1,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,40
2,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,32
3,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,13
4,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,1


## 2. The same split as phase 4

Train on 2011, test on 2012. Holding this fixed is what makes the two
runs below comparable -- change the split and the metrics stop meaning
anything relative to each other.

In [2]:
TARGET = "cnt"
FEATURES = [c for c in df.columns if c != TARGET]

train = df[df.yr == 0]
test = df[df.yr == 1]

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print(f"train {len(train)} rows (2011)   test {len(test)} rows (2012)")
print(f"{len(FEATURES)} features")

train 8645 rows (2011)   test 8734 rows (2012)
12 features


## 3. Baseline

An rmse of 126 means nothing on its own. Predicting the training mean
for every hour is the floor any real model has to clear.

In [3]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


def evaluate(y_true, y_pred):
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


baseline_pred = np.full(len(y_test), y_train.mean())
baseline = evaluate(y_test, baseline_pred)

print(f"baseline (predict the mean): rmse={baseline['rmse']:.1f} "
      f"mae={baseline['mae']:.1f} r2={baseline['r2']:.3f}")

baseline (predict the mean): rmse=227.8 mae=168.3 r2=-0.189


## 4. Two configurations

`min_samples_leaf` is the knob worth studying: it trades a little
accuracy for a much smaller artifact, which is what a serverless
endpoint cares about on cold start.

Run A is the phase 4 model. Run B grows deeper trees.

In [4]:
import joblib
from sklearn.ensemble import RandomForestRegressor

RUNS = {
    "A-leaf5": {"n_estimators": 100, "min_samples_leaf": 5},
    "B-leaf1": {"n_estimators": 100, "min_samples_leaf": 1},
}


def fit_and_score(params, seed=42):
    model = RandomForestRegressor(random_state=seed, n_jobs=-1, **params)
    model.fit(X_train, y_train)

    metrics = evaluate(y_test, model.predict(X_test))

    # Artifact size is a real deployment constraint, so measure it here
    # rather than discovering it at phase 8.
    buf = io.BytesIO()
    joblib.dump(model, buf)
    metrics["artifact_mb"] = buf.tell() / 1024 / 1024

    return model, metrics


results = {}
models = {}

for name, params in RUNS.items():
    models[name], results[name] = fit_and_score(params)
    m = results[name]
    print(f"{name}: rmse={m['rmse']:.1f} mae={m['mae']:.1f} "
          f"r2={m['r2']:.3f} size={m['artifact_mb']:.1f}MB")

A-leaf5: rmse=126.3 mae=89.4 r2=0.634 size=12.0MB


B-leaf1: rmse=125.2 mae=88.8 r2=0.641 size=70.7MB


## 5. Compare

In [5]:
table = pd.DataFrame(results).T
table.loc["baseline"] = {**baseline, "artifact_mb": 0.0}

table["rmse_vs_baseline"] = (
    (baseline["rmse"] - table["rmse"]) / baseline["rmse"] * 100
)

table = table[["rmse", "mae", "r2", "artifact_mb", "rmse_vs_baseline"]]
table.round(3)

,rmse,mae,r2,artifact_mb,rmse_vs_baseline
A-leaf5,126.350,89.417,0.634,11.965,44.537
B-leaf1,125.183,88.798,0.641,70.729,45.049
baseline,227.808,168.252,-0.189,0.000,0.000


## 6. Reproducibility

The phase 5 exit check. `random_state` is pinned, so refitting the same
configuration must land on identical metrics. If this assert fails, the
run-to-run comparison above is measuring noise rather than the
hyperparameter.

In [6]:
for name, params in RUNS.items():
    _, again = fit_and_score(params)

    for metric in ("rmse", "mae", "r2"):
        first, second = results[name][metric], again[metric]
        assert np.isclose(first, second), (
            f"{name} {metric} not reproducible: {first} != {second}"
        )

    print(f"{name}: reproducible")

print()
print("both runs reproducible -- metrics are the hyperparameter, not noise")

A-leaf5: reproducible


B-leaf1: reproducible

both runs reproducible -- metrics are the hyperparameter, not noise


## 7. Save the comparison

Phase 6 replays these two runs into MLflow. Writing them to S3 now means
that step compares against a record rather than a re-fit.

In [7]:
import json

payload = {
    "split": {"train_year": 2011, "test_year": 2012},
    "baseline": baseline,
    "runs": {name: {"params": RUNS[name], "metrics": results[name]}
             for name in RUNS},
}

body = json.dumps(payload, indent=2).encode()
s3.put_object(Bucket=BUCKET, Key="model/eval.json", Body=body)

print(f"s3://{BUCKET}/model/eval.json")
print(json.dumps(payload["runs"], indent=2)[:400])

s3://sagemaker-domain-dev-data-pqkx2l/model/eval.json
{
  "A-leaf5": {
    "params": {
      "n_estimators": 100,
      "min_samples_leaf": 5
    },
    "metrics": {
      "rmse": 126.34957564735046,
      "mae": 89.41737401151615,
      "r2": 0.634173693541707,
      "artifact_mb": 11.965119361877441
    }
  },
  "B-leaf1": {
    "params": {
      "n_estimators": 100,
      "min_samples_leaf": 1
    },
    "metrics": {
      "rmse": 125.182682517287


## 8. Notebook vs training job

Everything above ran on the space's instance -- billed for as long as the
app is up, whether or not a cell is running.

A SageMaker training job instead provisions a container, runs
`src/train.py`, writes the artifact to S3, and tears the instance down.
Nothing bills between jobs.

| | notebook | training job |
|---|---|---|
| billing | while the app is up | per job, to the second |
| iteration | instant, state in memory | ~3-4 min container start |
| record | cell outputs only | job name, params, metrics, logs |
| scales to | one instance | many, in parallel |

The rule of thumb: explore in the notebook, train in a job. Phase 7
turns the job into a pipeline step.

Submitting one is left to phase 7 rather than done here -- `src/submit_job.py`
targets the mlops bucket layout (`raw/bike/`, `models/`), not this stack's.